# 1. Copper equation of state

**Kernel:** MACE. **Before starting:** complete the preflight in `workshop_demo/README.md`.
Run cells from top to bottom in a fresh kernel. Each setup creates a new results directory.
Timings depend on the allocated hardware; the instructor should measure them before the session.

**Learning goals:** connect lattice spacing to energy; fit an equilibrium volume; distinguish a model prediction from an experimental measurement.

**Working pattern:** predict a result, run the calculation, inspect the geometry and convergence,
then explain the result to a partner. Energy is reported in eV, length in angstrom, and force in eV/angstrom.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in the repository, workshop folder, or exercise folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
WORKSHOP = next((p for base in _candidates for p in (base, base / 'workshop_demo')
                 if (p / 'workshop_utils.py').is_file()), None)
if WORKSHOP is None:
    raise RuntimeError('Launch Jupyter from the repository or workshop_demo folder.')
if str(WORKSHOP) not in sys.path:
    sys.path.insert(0, str(WORKSHOP))
from workshop_utils import start_exercise, mace_model, relax, smoke_check, signed_angle
DATA, OUTPUT = start_exercise('InorganicCrystals')
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.visualize import view


## 1. Construct a periodic crystal
ASE's `Atoms` stores geometry and boundary conditions. A calculator supplies energies and forces.
Predict: will compressing and expanding the crystal both increase its energy near equilibrium?
The primitive fcc cell below contains one atom, so its volume is also the volume per atom.


In [ ]:
from ase.build import bulk
from ase.eos import EquationOfState
from ase.units import kJ
atoms = bulk('Cu', 'fcc', a=3.615)
print('Atoms:', len(atoms), 'PBC:', atoms.pbc)
view(atoms.repeat((3, 3, 3)), viewer='x3d')


## 2. Attach the model and check one calculation
Model loading is separate from the energy scan so setup problems appear early.


In [ ]:
from mace.calculators import MACECalculator
DEVICE = 'cpu'  # Only select cuda inside a GPU allocation with a compatible environment.
calculator = MACECalculator(model_paths=mace_model('2023-12-03-mace-128-L1_epoch-199.model'),
                            device=DEVICE, default_dtype='float64')


In [ ]:
atoms.calc = calculator
smoke_check(atoms)


## 3. Sample the energy-volume curve
Start with nine points. The lattice scale changes all three lengths, so volume changes as scale cubed.
Keep a copy of the initial cell: scaling the already-scaled cell repeatedly would change the intended grid.


In [ ]:
scales = np.linspace(0.94, 1.06, 9)
cell0 = atoms.cell.copy()
volumes, energies = [], []
for scale in scales:
    atoms.set_cell(cell0 * scale, scale_atoms=True)
    volumes.append(atoms.get_volume())
    energies.append(atoms.get_potential_energy())
    print(f'{scale:.3f}: V={volumes[-1]:.4f} A^3, E={energies[-1]:.6f} eV')
np.savetxt(OUTPUT / 'energy_volume.csv', np.column_stack([volumes, energies]),
           delimiter=',', header='volume_A3,energy_eV', comments='')


## 4. Fit and inspect
The bulk modulus describes resistance to compression. A fit returning a number is not sufficient:
the data should bracket the minimum and the fitted curve should follow the sampled energies.


In [ ]:
eos = EquationOfState(volumes, energies, eos='birchmurnaghan')
V0, E0, B0 = eos.fit()
print(f'Equilibrium volume: {V0:.4f} A^3/atom')
print(f'fcc lattice parameter: {(4 * V0)**(1/3):.4f} A')
print(f'Bulk modulus: {B0 / kJ * 1e24:.2f} GPa')
print('Minimum bracketed:', min(volumes) < V0 < max(volumes))
eos.plot(filename=str(OUTPUT / 'Cu_EOS.png'), show=False)
plt.show()


## Try, explain, and report
1. Change the number of points to five. How stable are the fitted volume and modulus?
2. Narrow the scale range. Does the sampled minimum remain inside it?
3. Explain why a low force on a high-symmetry crystal does not prove that its cell volume is optimal.

**Checkpoint:** save your plot and report the model checkpoint, volume per atom, and bulk modulus.
Compare to an instructor-provided reference with a stated temperature and method; this calculation
is a static model energy fit, not a finite-temperature measurement.

**Continue:** oxygen adsorption uses the same calculator interface on a less symmetric structure.
